## 1. Data Wrangling
In this section we load the neural spike, trial, and neuron metadata datasets and perform preprocessing needed for analysis. Spike times are aggregated into firing rates for each neuron across trials to create a population activity matrix used for downstream decoding analyses.

In [ ]:
import numpy as np
import pandas as pd

# Load datasets (adjust paths if needed)
spikes = pd.read_csv("../data/spikes.csv")
trials = pd.read_csv("../data/trials.csv")
neurons = pd.read_csv("../data/neurons.csv")

# Inspect the data
print(spikes.head())
print(trials.head())
print(neurons.head())

# Time window for firing rate calculation
window_start = 0
window_end = 0.5

# Function to compute firing rate for a neuron in a trial
def compute_firing_rate(spike_times, start, end):
    spikes_in_window = spike_times[(spike_times >= start) & (spike_times <= end)]
    return len(spikes_in_window) / (end - start)

# Build population activity matrix
firing_rates = []

for trial in trials.itertuples():
    trial_rates = []
    for neuron_id in neurons['neuron_id']:
        neuron_spikes = spikes[spikes['neuron_id'] == neuron_id]['spike_time'].values
        rate = compute_firing_rate(neuron_spikes, trial.stim_onset + window_start, trial.stim_onset + window_end)
        trial_rates.append(rate)
    firing_rates.append(trial_rates)

# Convert to NumPy array
X = np.array(firing_rates)
y = trials['choice'].values

print("Population matrix shape:", X.shape)


## 2. Integrating Neural Activity with Anatomical Regions
In this section, neural activity is linked to anatomical brain regions using the neuron metadata provided in the dataset. Each recorded neuron is associated with a specific brain region, allowing us to examine how activity patterns vary across different parts of the brain. By merging neural firing rate data with the neuron metadata, we can group neurons by region and analyze whether certain brain areas contribute more strongly to the decoding of behavioral choices. This integration enables us to explore how distributed neural populations across the brain encode information related to decision-making.

In [ ]:
region_labels = neurons['region'].values

# Convert population activity to DataFrame
activity_df = pd.DataFrame(X)
activity_df.columns = neurons['neuron_id']
activity_df['choice'] = y

print(activity_df.head())

## 3. Dimensionality Reduction (PCA)
Neural population recordings often contain activity from many neurons simultaneously, resulting in high-dimensional datasets that can be difficult to visualize and analyze directly. Principal Component Analysis (PCA) is a dimensionality reduction technique that identifies patterns in the data by finding directions (principal components) that capture the greatest variance in neural activity. By projecting the population activity matrix onto the first few principal components, we can reduce the complexity of the dataset while preserving the most important information. This allows us to visualize neural population activity in two dimensions and explore whether patterns in neural activity relate to behavioral variables such as decision outcomes.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='coolwarm', alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Neural Population Activity PCA")
plt.colorbar(label='Behavioral Choice')
plt.show()

## 4. Decoding Behavioral Choice
Neural population recordings often contain activity from many neurons simultaneously, resulting in high-dimensional datasets that can be difficult to visualize and analyze directly. Principal Component Analysis (PCA) is a dimensionality reduction technique that identifies patterns in the data by finding directions (principal components) that capture the greatest variance in neural activity. By projecting the population activity matrix onto the first few principal components, we can reduce the complexity of the dataset while preserving the most important information. This allows us to visualize neural population activity in two dimensions and explore whether patterns in neural activity relate to behavioral variables such as decision outcomes.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='coolwarm', alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Neural Population Activity PCA")
plt.colorbar(label='Behavioral Choice')
plt.show()

## 5. Comparing Brain Regions
Because neurons were recorded from multiple brain regions, we next examine whether certain regions contribute more strongly to decoding behavioral outcomes. To do this, neural activity is grouped according to the anatomical region associated with each neuron using the neuron metadata. Separate decoding analyses are then performed for neurons from each brain region. By comparing decoding performance across regions, we can evaluate whether specific areas of the brain contain stronger representations of task-related information. This analysis provides insight into how information relevant to decision-making may be distributed across different parts of the brain.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000)

accuracy = cross_val_score(model, X, y, cv=5, scoring='accuracy')
auc = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print("Mean accuracy:", accuracy.mean())
print("Mean AUC:", auc.mean())

## 6. Permutation Testing
To determine whether decoding performance is significantly above chance, we perform a permutation test. In this procedure, the behavioral labels (e.g., choices) are randomly shuffled across trials while the neural activity data remain unchanged. The decoding model is then retrained on this shuffled dataset multiple times to generate a distribution of accuracy values expected by chance. By comparing the true decoding accuracy to this null distribution, we can assess whether the observed performance is significantly greater than what would be expected if neural activity and behavior were unrelated.

In [ ]:
region_performance = {}

for region in neurons['region'].unique():
    region_neurons = neurons[neurons['region'] == region]['neuron_id']
    region_X = activity_df[region_neurons].values
    
    if region_X.shape[1] > 2:  # need at least 3 neurons to train
        scores = cross_val_score(model, region_X, y, cv=5)
        region_performance[region] = scores.mean()

print("Region-specific decoding accuracy:")
for region, acc in region_performance.items():
    print(f"{region}: {acc:.3f}")